XGBoost (eXtreme Gradient Boosting) is a fast, regularized implementation of gradient-boosted decision trees. It comes from the 2016 paper "XGBoost: A Scalable Tree Boosting System" by Tianqi Chen and Carlos Guestrin.

## 1. Core idea:

The boosting loop is the one from your gradient boosting notebooks: start with a constant prediction, then repeatedly add a small tree, so that F_m(x) = F_(m−1)(x) + η · f_m(x), where η is the learning rate. XGBoost differs in **how each tree is built**: it optimizes a regularized objective, using both the first and second derivatives of the loss.

## 2. The math:

At each round, XGBoost minimizes:

**Obj = Σᵢ l(yᵢ, ŷᵢ) + Σₖ Ω(fₖ)**, where **Ω(f) = γT + ½ λ Σⱼ wⱼ²** (plus an optional L1 term α Σ|wⱼ|)

T is the number of leaves and wⱼ are the leaf values. So the model is penalized for having many leaves and for large leaf values.

It then approximates the loss with a second-order Taylor expansion. For each sample i it computes a gradient **gᵢ = ∂l/∂ŷ** and a hessian **hᵢ = ∂²l/∂ŷ²**:

| Loss | gᵢ | hᵢ |
|---|---|---|
| Squared error | ŷ − y | 1 |
| Log-loss (classification) | p − y | p(1 − p) |

For a leaf containing a set of samples, let G = Σ gᵢ and H = Σ hᵢ. Then:

- **Optimal leaf value:** w* = −G / (H + λ)
- **Gain of a split:** Gain = ½ [ G_L²/(H_L+λ) + G_R²/(H_R+λ) − (G_L+G_R)²/(H_L+H_R+λ) ] − γ

A split is only made if its gain is positive, so **γ acts as built-in pruning**.

This connects directly to my gradient boosting classification notebooks. There, the leaf value was Σ residuals / Σ p(1−p). That is exactly −G/H with λ = 0 (the sign flips because residual = y − p = −g). XGBoost's λ adds shrinkage to that value, and it also uses G and H to choose the splits, not just to set leaf values.

## 3. How XGBoost finds splits:

- **Exact greedy:** sort each feature and scan every possible threshold. It's accurate but slow and memory-heavy on big data.
- **Histogram method (`hist`):** bucket each feature into at most `max_bin` bins (256 by default), build gradient and hessian histograms, and scan the bins. The docs say `auto` now means the same as `hist`, so this is what you get by default.
- **Histogram subtraction:** a child's histogram is the parent's minus its sibling's, which saves about half the work.
- **Weighted quantile sketch (`approx`):** chooses candidate split points by hessian-weighted quantiles instead of raw counts.
- **Sparsity-aware splits:** each split learns a **default direction** for missing or zero values, so missing data needs no imputation.
- **Growth policy:** `depthwise` grows level by level (the default). `lossguide` (leaf-wise) always splits the leaf with the best gain and is controlled by `max_leaves`.

## 4. System-level optimizations:

- Data is stored in sorted, compressed **column blocks** and reused across trees, so sorting isn't repeated every round.
- Memory access is **cache-aware**, and split finding is **parallelized across features** (the trees themselves stay sequential).
- **Out-of-core and external-memory** training handles data larger than RAM.
- **GPU training** parallelizes split finding. In the docs, GPU histogram growing is enabled by using `hist` together with `device=cuda`.
- **Distributed training** works through Dask and Spark.

Recent releases keep extending this:

- 3.1 added a categorical re-coder, an adaptive cache for CUDA external memory, and faster CPU inference through array-based tree traversal.
- 3.2 added more progress on multi-target trees with vector leaves, plus better GPU external-memory training. The vector leaf is still marked as work in progress.

## 5. Overfitting controls:

| Parameter | Default | Effect |
|---|---|---|
| `eta` / `learning_rate` | 0.3 | Shrinks each tree's contribution. Lower is safer but needs more trees |
| `max_depth` | 6 | Tree depth. Deeper means more interactions and more overfitting |
| `min_child_weight` | 1 | Minimum hessian sum per child. Higher is more conservative |
| `gamma` (`min_split_loss`) | 0 | Minimum gain needed to split |
| `lambda` (`reg_lambda`) | 1 | L2 penalty on leaf values |
| `alpha` (`reg_alpha`) | 0 | L1 penalty on leaf values |
| `subsample` | 1 | Fraction of rows per tree |
| `colsample_bytree` / `bylevel` / `bynode` | 1 | Fraction of columns per tree, level or node |

Row and column subsampling come from bagging and random forests, so XGBoost combines boosting with some variance reduction.

## 6. Performance:

- **Versus classic gradient boosting** (scikit-learn's `GradientBoostingClassifier`): XGBoost is typically much faster, because of histograms, multithreading and GPU support. It's also better regularized and handles missing values natively.
- **Accuracy:** it is strong on tabular data and has been a staple of tabular competitions. Its edge over plain GBM comes mostly from regularization and tuning.
- **Versus LightGBM and CatBoost** (general tendencies, not measured here): LightGBM is often a little faster on very large datasets, and CatBoost often handles high-cardinality categorical features better out of the box. On a given dataset any of the three can win, so try more than one.
- **Where time goes:** training time scales roughly with rows × features × trees, and with `max_depth` and `max_bin`. Inference time scales with the number of trees times their depth.

## 7. Optimization points:

**Speed and memory**
- Use `tree_method="hist"` (already the default) and `device="cuda"` if you have a GPU.
- Lower `max_bin` (for example 64 or 128) for a speed gain with usually negligible accuracy loss.
- Use `QuantileDMatrix` to reduce memory with `hist`, and float32 numeric data.
- Use `subsample` and `colsample_*` below 1, which cut work per tree and also regularize.
- Use external memory when the data exceeds RAM.
- Keep inference lean: fewer, shallower trees mean lower latency.

**Accuracy**
- Use a low `learning_rate` (0.01 to 0.1) with many trees, and let **early stopping** pick the tree count.
- Tune `max_depth` and `min_child_weight` first, then `subsample` / `colsample_*`, then `lambda`, `alpha` and `gamma`.
- Pick an evaluation metric that matches the goal, such as `aucpr` for imbalanced data.

**Imbalanced classes**
- Set `scale_pos_weight`. The docs suggest a typical starting value of the sum of negative instances divided by the sum of positive instances.
- Prefer ranking metrics such as AUC-PR over plain accuracy.

**Other useful features**
- Native categorical support with `enable_categorical=True` (with `hist`).
- `monotone_constraints` and `interaction_constraints` to encode domain knowledge.
- Built-in cross-validation with `xgb.cv`, and SHAP-style contributions through `pred_contribs`.

## 8. Practical tuning order:

1. Fix `learning_rate` around 0.05 to 0.1 and set a large `n_estimators` with early stopping on a validation set.
2. Tune `max_depth` (3 to 8) and `min_child_weight` (1 to 10).
3. Tune `subsample` and `colsample_bytree` (0.6 to 1.0).
4. Tune `reg_lambda`, `reg_alpha` and `gamma`.
5. Lower the learning rate, retrain, and confirm the gain on a held-out test set.

## 9. Minimal code:

```python
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    tree_method="hist",
    eval_metric="logloss",
    early_stopping_rounds=50,
    random_state=42,
)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
print(model.best_iteration, model.best_score)
```

## 10. Limitations:

- It has many hyperparameters, and untuned defaults can overfit small or noisy datasets.
- It is sensitive to label noise, as all boosting is.
- It can't extrapolate beyond the range of the training targets, like any tree model.
- It is less interpretable than a single tree, though feature importance and SHAP help.
- On unstructured data (images, text, audio), neural networks usually win.

---